# Companies House - Data Import and Construction Script

This script imports the latest **Basic Company Data** snapshot for *live* companies from the [Free Company Data Product (FCDP)](https://download.companieshouse.gov.uk/en_output.html), along with the latest People with Significant Control (PSC) snapshot from [PSC Data Product](https://download.companieshouse.gov.uk/en_pscdata.html).

Some helper functions have been defined in *utils.py* and *visualisations.py* to ensure that this script is shorter. A lot of the notes regarding steps taken have been left inside the code cells within this script to keep it shorter vs placing extra notes within markdown cells.

- The *fcdp* dataset is provided as a *csv* file containing one row per company.
- The *PSC* dataset is provided as a *txt* file in *JSON* format and contains multiple records per company.
<br><br>

To create a usable analytical dataset:
- both sources are reduced to a selected set of relevant columns,
- an age based predictor column is feature engineered *(company_age_when_acc_due)*
- along with the target variable "overdue" *(**1,0**)*
- the *PSC* data is aggregated to give **1** row per company *(psc_count, has_corporate_psc, has_foreign_psc, has_sanctioned_psc, recent_psc_change (within last **12** months))*
- the datasets are joined together on company number
- then the final dataset is saved in *Parquet* format for use in further analysis in the *CH_EDA_and_Cleansing.pynb*.

As both datasets are large, all the processed outputs are saved locally in *Parquet* format, allowing other scripts to load the data efficiently without repeating the full import and transformation process.

Encoding of categorical variables will be conducted after some EDA in the *CH_EDA_and_Cleansing.pynb*.

In [1]:
# ---------------------------------------------------------
# 1. Library imports
# ---------------------------------------------------------

# Standard library
import os                                         # File and directory operations
import sys                                        # Gives access to Python's system-level variables and functions
sys.path.append(os.path.abspath(".."))            # Needed so the notebook can find src/

import warnings
warnings.filterwarnings("ignore") # suppress non-critical warnings to keep the notebook clean.

# System libraries
import platform          # For displaying operating system, python version used and CPU processor used when running the code
import psutil            # For displaying CPU cores and RAM on computer used when running the code           
import importlib         # For displaying the library versions used when running the code

# Core data and query engines
import duckdb        # High performance SQL engine for local analytics - This is used for speed.
import pandas as pd  # DataFrames, joins, lightweight transforms

# Utilities
from datetime import date     # For date stamping outputs
from datetime import datetime # For converting date columns
import time                   # For measuring execution timings of steps
from pathlib import Path      # Modern, OS-agnostic filesystem path handling

# Import src helper functions
from src import (
    system_summary,                  # System and library summary information helper (utils.py)
    find_project_root,               # Dataset summary helper (utils.py)
    mask_user,                       # Masking the username helper (utils.py)
    drop_columns_and_report          # Function for dropping columns and displaying information (utils.py)
)

In [2]:
# Display information about the operating system etc plus python library versions used when running this project.
print(f" Basic system and Python library information:")
system_summary(libraries=[
    "duckdb",              # used in get data script
    "pandas",              # used in all scripts
    "numpy",               # used in EDA and cleansing script & logistic regression script
    "seaborn",             # used in EDA and cleansing script & logistic regression script
    "matplotlib",          # used in EDA and cleansing script & logistic regression script
    "statsmodels",         # used in EDA and cleansing script
    "scipy",               # used in EDA and cleansing script
    "sklearn",             # used in logistic regression script
    "imblearn"             # used in logistic regression script
])

 Basic system and Python library information:


{'Operating System': 'Windows-10-10.0.26200-SP0',
 'Python Version': '3.11.13',
 'CPU Processor': 'Intel64 Family 6 Model 186 Stepping 2, GenuineIntel',
 'CPU Physical Cores': 14,
 'CPU Logical Cores': 20,
 'RAM (GB)': 63.69,
 'duckdb version': '1.5.2',
 'pandas version': '2.2.3',
 'numpy version': '2.1.3',
 'seaborn version': '0.13.2',
 'matplotlib version': '3.10.0',
 'statsmodels version': '0.14.4',
 'scipy version': '1.15.3',
 'sklearn version': '1.8.0',
 'imblearn version': '0.14.1'}

In [3]:
# ---------------------------------------------------------
# 2. File paths
# ---------------------------------------------------------

# Set the main project folder, the base directory for the project
# All other paths are built from this location.
BASE_DIR = find_project_root(Path.cwd())

# Set up the folders used in the project.
# 'data_raw' holds the original Companies House files.
# 'data_processed' stores the cleaned and converted outputs.
data_dir = BASE_DIR / "Companies_House"
raw_dir = data_dir / "data_raw"
processed_dir = data_dir / "data_processed"

# Make sure the processed-data folder exists, if it doesn't then create it.
processed_dir.mkdir(parents=True, exist_ok=True)

# Paths to the raw input files downloaded from Companies House.
fcdp_path = raw_dir / "BasicCompanyDataAsOneFile-2026-03-02.csv"
psc_path = raw_dir / "persons-with-significant-control-snapshot-2026-03-12.txt"

# Paths to the cached Parquet outputs created during the ETL process.
# These speed up future runs by avoiding repeated csv loading.
fcdp_parquet = processed_dir / "fcdp.parquet"
psc_parquet = processed_dir / "psc.parquet"
fcdp_selected_path = processed_dir / "fcdp_selected.parquet"
psc_features_path = processed_dir / "psc_features.parquet"
final_output_path = processed_dir / "company_psc_dataset.parquet"   

# Create an in-memory DuckDB connection, this is used for fast SQL queries on large datasets.
con = duckdb.connect()

This next section takes **~17seconds** the first time it runs, while it creates the parquet file. Subsequent runs when a parquet file already exists take **~10seconds**.

In [4]:
# ---------------------------------------------------------
# 3. FCDP load (via DuckDB to Parquet to pandas)
# ---------------------------------------------------------

start = time.time()
# If a Parquet version of the FCDP data already exists AND is not empty, then load it directly,
# this avoids re-reading the large raw file.
# Print a checkpoint label to tell users what is happening.
if  fcdp_parquet.exists() and fcdp_parquet.stat().st_size > 0:
    print(f"Loading FCDP from Parquet")
    fcdp = pd.read_parquet(fcdp_parquet)
else:
    # If the file does not exist then create it from the raw file,
    print(f"Converting FCDP CSV -> Parquet (first time only)")
    if not fcdp_path.exists():
        raise FileNotFoundError(f"Source file not found: {fcdp_path}")

    # Use DuckDB to read the raw csv file and write it to parquet
    # DuckDB uses SQL and does the heavy lifting of reading the huge file and converting it to parquet.
    # Pandas is still required for working with the final in-memory dataset.
    # The EDA, clustering and modelling all expect a pandas dataframe,
    # for (sci-kit learn, seaborn, matplotlib, statsmodels, plotly)
    con.execute(f"""
        COPY (
            SELECT *
            FROM read_csv_auto('{fcdp_path}')
        )
        TO '{fcdp_parquet}' (FORMAT PARQUET);
    """)

    # Check that the Parquet file was created successfully.
    if not fcdp_parquet.exists() or fcdp_parquet.stat().st_size == 0:
        raise RuntimeError(f"Failed to create valid FCDP Parquet file")

    # Load the new fcdp parquet file into pandas.
    fcdp = pd.read_parquet(fcdp_parquet)

# Print the number of rows that were loaded so the user knows that the load was successful
print(f"FCDP rows: {len(fcdp):,}")
# Calculate the amount of time the process takes
end = time.time()
print(f"FCDP Load time: {end - start:.3f} seconds")

Converting FCDP CSV -> Parquet (first time only)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FCDP rows: 5,677,276
FCDP Load time: 49.354 seconds


This next section takes **~37seconds** the first time it runs, while it creates the parquet file. Subsequent runs when a parquet file already exists take **~10seconds**.

In [5]:
# ---------------------------------------------------------
# 4. PSC load (via DuckDB to Parquet)
# ---------------------------------------------------------
start = time.time()
# If a Parquet version of the PSC data already exists AND it is not empty, then load it directly.
# print a checkpoint label to tell users what is happening.
if psc_parquet.exists() and psc_parquet.stat().st_size > 0:
    print(f"PSC Parquet already exists (will use DuckDB directly from it)")
else:
    # If the file didn't exist or it was empty then create it, but first print a checkpoint label to tell users what is happening.
    print(f"Converting PSC NDJSON -> Parquet (first time only)")
    if not psc_path.exists():
        raise FileNotFoundError(f"Source file not found: {psc_path}")

        # Use DuckDB to read the PSC JSON file and write it to parquet
        # Because JSON data is nested we:
        # - sample a large number of rows to infer the schema
        # - union fields by name
        # - ignore malformed rows instead of crashing
        # DuckDB then writes the resulting dataset directly to Parquet.
    con.execute(f"""
        COPY (
            SELECT *
            FROM read_json_auto(
                '{psc_path}',
                sample_size=100000,
                union_by_name=true,
                ignore_errors=true
            )
        )
        TO '{psc_parquet}' (FORMAT PARQUET);
    """)

    # Verify that the Parquet file was created successfully
    if not psc_parquet.exists() or psc_parquet.stat().st_size == 0:
        raise RuntimeError(f"Failed to create valid PSC Parquet file")

# Don't load this dataset into pandas as it is too large and needs aggregating first using DuckDB from step 8.
# Print a checkpoint message for user benefit.
print(f"PSC Parquet ready")
# Calculate the amount of time the process takes
end = time.time()
print(f"PSC Test time: {end - start:.10f} seconds")

Converting PSC NDJSON -> Parquet (first time only)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

PSC Parquet ready
PSC Test time: 137.5481159687 seconds


In [6]:
# ---------------------------------------------------------
# 5. FCDP COLUMN SELECTION + STANDARDISATION
# ---------------------------------------------------------
# At this stage, the raw FCDP dataset has been loaded into pandas.
# In this step:
#   - check which columns are available
#   - keep only the fields needed for analysis
#   - rename them to clean, consistent snake_case
#   - prepare the dataset for feature engineering (age, dissolved_flag)
# ---------------------------------------------------------

# Print a list of all available columns in the fcdp dataset to see what is available
print(f"FCDP columns:")
for col in fcdp.columns:
    print(f" -", col)

# Map the raw column names to cleaner snake_case names for the chosen columns.
column_map = {
    "CompanyNumber": "company_number",                # Company identifier
    "IncorporationDate": "incorporation_date",        # Date the company was formed
    "CompanyCategory": "company_category",            # Legal form (Ltd, LLP, etc...)
    "Accounts.NextDueDate": "account_due_date",       # When the next accoutns are due
    "Accounts.AccountCategory": "accounts_category",  # Filing category (micro, small, etc...)
    "SICCode.SicText_1": "industry",                  # Primary SIC industry description
    "CountryOfOrigin": "country_of_origin",           # Country where the company was formed
    "RegAddress.Country": "registered_country",       # Country of the registered office.
}

# Select only the columns required and rename them.
fcdp_selected = fcdp.loc[:, column_map.keys()].rename(columns=column_map).copy()

# Display the top 5 rows
fcdp_selected.head()

FCDP columns:
 - CompanyName
 - CompanyNumber
 - RegAddress.CareOf
 - RegAddress.POBox
 - RegAddress.AddressLine1
 - RegAddress.AddressLine2
 - RegAddress.PostTown
 - RegAddress.County
 - RegAddress.Country
 - RegAddress.PostCode
 - CompanyCategory
 - CompanyStatus
 - CountryOfOrigin
 - DissolutionDate
 - IncorporationDate
 - Accounts.AccountRefDay
 - Accounts.AccountRefMonth
 - Accounts.NextDueDate
 - Accounts.LastMadeUpDate
 - Accounts.AccountCategory
 - Returns.NextDueDate
 - Returns.LastMadeUpDate
 - Mortgages.NumMortCharges
 - Mortgages.NumMortOutstanding
 - Mortgages.NumMortPartSatisfied
 - Mortgages.NumMortSatisfied
 - SICCode.SicText_1
 - SICCode.SicText_2
 - SICCode.SicText_3
 - SICCode.SicText_4
 - LimitedPartnerships.NumGenPartners
 - LimitedPartnerships.NumLimPartners
 - URI
 - PreviousName_1.CONDATE
 - PreviousName_1.CompanyName
 - PreviousName_2.CONDATE
 - PreviousName_2.CompanyName
 - PreviousName_3.CONDATE
 - PreviousName_3.CompanyName
 - PreviousName_4.CONDATE
 - Previ

,company_number,incorporation_date,company_category,account_due_date,accounts_category,industry,country_of_origin,registered_country
0,08209948,2012-09-11,Private Limited Company,2026-06-30,DORMANT,99999 - Dormant Company,United Kingdom,ENGLAND
1,11743365,2018-12-28,Private Limited Company,2026-09-30,DORMANT,59112 - Video production activities,United Kingdom,None
2,16873705,2025-11-25,Private Limited Company,2027-08-25,NO ACCOUNTS FILED,86210 - General medical practice activities,United Kingdom,UNITED KINGDOM
3,15073164,2023-08-15,Private Limited Company,2026-12-31,TOTAL EXEMPTION FULL,70229 - Management consultancy activities othe...,United Kingdom,ENGLAND
4,13522064,2021-07-21,Private Limited Company,2027-04-30,MICRO ENTITY,58290 - Other software publishing,United Kingdom,UNITED KINGDOM


In [7]:
# ---------------------------------------------------------
# 6. COMPANY FEATURE ENGINEERING
# ---------------------------------------------------------
# Now that FCDP has been cleaned and standardised, we create
# two new engineered features:
#   - company_age_when_acc_due: the age of the company in years when the accounts are due
#   - overdue: target variable (1=filed late, 0=filed on time)
# ---------------------------------------------------------

# Companies house data is a snapshot. It shows filing status as of the
# date in the filename, not today's date. So we extract that date and
# use it as the reference point for the overdue flag.

# Extract snapshot date from the filename
fcdp_filename = fcdp_path.stem  # "BasicCompanyDataAsOneFile-2026-03-02"
snapshot_str = fcdp_filename.split("-")[-3:]  # ["2026", "03", "02"]

snapshot_date = pd.Timestamp("-".join(snapshot_str))
snapshot_date


# Make sure the date columns are proper proper datetime types
fcdp_selected["incorporation_date"] = pd.to_datetime(fcdp_selected["incorporation_date"], errors="coerce")
fcdp_selected["account_due_date"] = pd.to_datetime(fcdp_selected["account_due_date"], errors="coerce")

# 1. Use incorporation date and account_due_date to calculate the company age at the time accounts are due
fcdp_selected["company_age_when_acc_due"] = (
    (fcdp_selected["account_due_date"] - fcdp_selected["incorporation_date"])
    .dt.days / 365.25
)

# 2. Target variable: Overdue at the snapshot date
fcdp_selected["overdue"] = (snapshot_date > fcdp_selected["account_due_date"]).astype(int)

# Remove date columns and company_status column as they are no longer needed, using the 'drop_columns_and_report' helper function from 'utils.py'
fcdp_selected = drop_columns_and_report(fcdp_selected, columns=["incorporation_date", "account_due_date"])

print(f"Added company_age_when_acc_due and overdue target variable to FCDP dataset")

rows: 5,677,276
columns: 8
Added company_age_when_acc_due and overdue target variable to FCDP dataset


In [8]:
# Display the top 5 rows of the data
fcdp_selected.head()

,company_number,company_category,accounts_category,industry,country_of_origin,registered_country,company_age_when_acc_due,overdue
0,08209948,Private Limited Company,DORMANT,99999 - Dormant Company,United Kingdom,ENGLAND,13.798768,0
1,11743365,Private Limited Company,DORMANT,59112 - Video production activities,United Kingdom,None,7.756331,0
2,16873705,Private Limited Company,NO ACCOUNTS FILED,86210 - General medical practice activities,United Kingdom,UNITED KINGDOM,1.746749,0
3,15073164,Private Limited Company,TOTAL EXEMPTION FULL,70229 - Management consultancy activities othe...,United Kingdom,ENGLAND,3.378508,0
4,13522064,Private Limited Company,MICRO ENTITY,58290 - Other software publishing,United Kingdom,UNITED KINGDOM,5.774127,0


In [9]:
# Count how many companies are overdue vs not overdue.
# Convert the result into a small table with:
# - the number of companies in each group
# - the percentage share of each group.
fcdp_selected["overdue"].value_counts().to_frame("count").assign(
    percentage=lambda x: (x["count"] / x["count"].sum() * 100).round(2)
)

,count,percentage
overdue,,
0,5235729,92.22
1,441547,7.78


In [10]:
# ---------------------------------------------------------
# 7. SAVE CLEANED FCDP DATASET (for downstream scripts)
# ---------------------------------------------------------
# At this point:
#   - The raw FCDP file has been loaded
#   - Only the required columns have been selected
#   - Column names have been standardised
#   - New features have been added
#
# This step saves the cleaned FCDP subset to Parquet so that
# other scripts can load it instantly without repeating the
# expensive and slow CSV -> pandas transformation.
# The user name is masked.
# ---------------------------------------------------------

fcdp_selected.to_parquet(fcdp_selected_path, index=False)
print(f"Saved cleaned FCDP dataset to {mask_user(str(fcdp_selected_path))} on {date.today()}")

Saved cleaned FCDP dataset to C:\Users\******\Documents\DS_Assignments\Companies_House\data_processed\fcdp_selected.parquet on 2026-05-10


In [11]:
# ---------------------------------------------------------
# 8. PSC STRUCTURE INSPECTION (understand nested JSON fields)
# ---------------------------------------------------------
# PSC data is deeply nested JSON stored inside a 'data' column.
# Before flattening or selecting fields, we inspect a single row
# to understand the top-level keys available inside the PSC object.
# This avoids loading the full dataset and helps guide the flattening step.
# ---------------------------------------------------------

# Load one PSC row to inspect the structure
sample_psc = con.execute(f"""
    SELECT data
    FROM read_parquet('{psc_parquet}')
    LIMIT 1
""").fetchone()[0]

# Print the top-level keys inside the PSC 'data' object
print(f"Top-level PSC data fields:")
for key in sample_psc.keys():
    print(f" -", key)

Top-level PSC data fields:
 - address
 - etag
 - identification
 - kind
 - links
 - name
 - natures_of_control
 - notified_on
 - country_of_residence
 - date_of_birth
 - identity_verification_details
 - name_elements
 - nationality
 - ceased_on
 - ceased
 - description
 - is_sanctioned
 - principal_office_address


This next section takes **~18seconds**.

In [12]:
# ---------------------------------------------------------
# 9. PSC PROCESSING (flatten, add recent flags, aggregate the data)
# ---------------------------------------------------------
# The PSC dataset contains nested JSON and multiple rows per company.
# In this step we:
#   1. Read the PSC Parquet into DuckDB as a table
#   2. Flatten the nested JSON structure into simple columns
#   3. Create flags for recent PSC activity (added/closed in last 12 months)
#   4. Aggregate PSC information to one row per company
#
# This produces a compact PSC feature table that can be joined
# to the cleansed FCDP company dataset in the next step.
# ---------------------------------------------------------
start = time.time()
# Step 9.1 — Create a DuckDB table over the PSC Parquet, remember DuckDB uses SQL.
# Use snake_case for the column headers.
print(f"PSC Step 1: Creating DuckDB table from Parquet...")
con.execute(f"""
    CREATE OR REPLACE TABLE psc AS
    SELECT
        company_number,                                          -- Company identifier
        data['kind']::VARCHAR AS kind,                           -- PSC type
        TRY_CAST(data['notified_on'] AS DATE) AS notified_on,    -- PSC added
        TRY_CAST(data['ceased_on'] AS DATE) AS ceased_on,        -- PSC ceased
        CAST(data['is_sanctioned'] AS BOOLEAN) AS is_sanctioned, -- Sanctions flag
        LOWER(data['address']['country']) AS psc_country         -- Country normalised
    FROM read_parquet('{psc_parquet}');
""")
print(f"PSC table created")

# Step 9.2 — Flatten PSC fields + compute recent flags
print(f"PSC Step 2: Flattening PSC fields...")
con.execute(f"""
    CREATE OR REPLACE TABLE psc_flat AS
    SELECT
        company_number,
        kind,
        notified_on,
        ceased_on,
        is_sanctioned,
        psc_country,

        -- Recent PSC activity (last 12 months)
        (notified_on >= CURRENT_DATE - INTERVAL 12 MONTH) AS recent_added,
        (ceased_on   >= CURRENT_DATE - INTERVAL 12 MONTH) AS recent_ceased

    FROM psc;
""")
print(f"PSC flattening complete")

# Step 9.3 — Aggregate PSC features to company level
# Count the number of PSCs per company
# FLag 1 when at least one PSC is a corporate entity
# Flag 1 when at least one PSC is not UK based
# Flag 1 when at least one PSC is sanctioned
# Flag 1 when at least one PSC has recently been added or left, there has been a recent change
print(f"PSC Step 3: Aggregating PSC features to company level...")
psc_features_df = con.execute(f"""
    SELECT
        company_number,                                 -- One row per company
        COUNT(*) AS psc_count,                          -- Total PSCs

        MAX(CASE WHEN kind = 'corporate-entity-person-with-significant-control'
                 THEN 1 ELSE 0 END) AS has_corporate_psc,

        MAX(CASE WHEN psc_country IS NOT NULL
                  AND psc_country NOT IN ('united kingdom', 'uk')
                 THEN 1 ELSE 0 END) AS has_foreign_psc,

        MAX(CASE WHEN is_sanctioned THEN 1 ELSE 0 END) AS has_sanctioned_psc,

        MAX(CASE WHEN recent_added OR recent_ceased THEN 1 ELSE 0 END)
            AS recent_psc_change

    FROM psc_flat
    GROUP BY company_number;
""").df()
print(f"PSC aggregation complete")

# Calculate how long this process took
end = time.time()
print(f"PSC Flattening and Aggregation time: {end - start:.3f} seconds")

# Display the first 5 rows of data.
print(f"Preview of PSC features:")
psc_features_df.head()

PSC Step 1: Creating DuckDB table from Parquet...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

PSC table created
PSC Step 2: Flattening PSC fields...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

PSC flattening complete
PSC Step 3: Aggregating PSC features to company level...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

PSC aggregation complete
PSC Flattening and Aggregation time: 63.124 seconds
Preview of PSC features:


,company_number,psc_count,has_corporate_psc,has_foreign_psc,has_sanctioned_psc,recent_psc_change
0,OC366807,3,0,0,0,0
1,10303236,2,1,1,0,0
2,09732577,1,0,1,0,0
3,11202785,1,0,1,0,0
4,SC589484,2,1,1,0,0


In [13]:
# ---------------------------------------------------------
# 10. SAVE PSC FEATURES (for downstream scripts)
# ---------------------------------------------------------
# At this point:
#   - PSC has been flattened
#   - Recent activity flags have been computed
#   - Company‑level PSC features have been aggregated
#
# This step saves the PSC feature table to Parquet so that
# later notebooks/scripts can load it instantly without
# repeating the expensive PSC processing pipeline.
# The username is masked.
# ---------------------------------------------------------

psc_features_df.to_parquet(psc_features_path, index=False)
print(f"Saved PSC features to {mask_user(str(psc_features_path))} on {date.today()}")

Saved PSC features to C:\Users\******\Documents\DS_Assignments\Companies_House\data_processed\psc_features.parquet on 2026-05-10


In [14]:
# ---------------------------------------------------------
# 11. JOIN FCDP + PSC FEATURES = FINAL COMPANIES HOUSE DATASET
# ---------------------------------------------------------
# In this step we:
#   - Load the cleaned FCDP dataset (company-level fields)
#   - Load the aggregated PSC feature table (PSC-level signals)
#   - Perform a LEFT JOIN to retain all companies, even those
#     without PSC records (PSC is optional for many firms)
#   - Save the final Companies House dataset to Parquet
#
# This produces a unified, analysis-ready dataset combining
# company attributes + PSC risk indicators.
# ---------------------------------------------------------
start = time.time()
print(f"Loading processed FCDP and PSC feature tables...")

# Load the pre-processed datasets from Parquet
fcdp_selected = pd.read_parquet(fcdp_selected_path)
psc_features  = pd.read_parquet(psc_features_path)

print(f"FCDP rows: {len(fcdp_selected):,}")
print(f"PSC feature rows: {len(psc_features):,}")

# Validate join keys
if fcdp_selected["company_number"].duplicated().any():
    raise ValueError("Duplicate company_number in fcdp_selected")

if psc_features["company_number"].duplicated().any():
    raise ValueError("Duplicate company_number in psc_features")

# ---------------------------------------------------------
# Perform the join
# ---------------------------------------------------------
# Left join keeps all companies from FCDP, even if they have no PSC data.
final_dataset = fcdp_selected.merge(
    psc_features,
    on="company_number",
    how="left"
)

# Reorder columns into a sensible order

def reorder_columns(df):
    ordered_cols = [
        "company_number",           # Unique Companies House ID
        "company_category",         # Company type (Ltd, LLP, etc.)
        "company_age_when_acc_due", # Company age when the next account is due to be filed.
        "accounts_category",        # Accounts filing category (micro, small, full)
        "industry",                 # SIC-derived industry/sector
        "country_of_origin",        # Country where company was incorporated
        "registered_country",       # Country where company is currently registered
        "psc_count",                # Number of PSCs
        "has_corporate_psc",        # True if any PSC is a corporate entity
        "has_foreign_psc",          # True if any PSC is non‑UK
        "has_sanctioned_psc",       # True if any PSC appears on sanctions lists
        "recent_psc_change",        # PSC changes in recent period
        "overdue"                   # Target variable (1 = overdue)
    ]
    return df[ordered_cols]

final_dataset = reorder_columns(final_dataset)

# Randomly shuffle the dataset
final_dataset = final_dataset.sample(frac=1, random_state=42).reset_index(drop=True)
# Replace company_number with a new monotonic ID
final_dataset["company_number"] = final_dataset.index + 1

# PSC columns that should be integer end up being float due to the way pandas behaves in the join, so convert them back to integer.
psc_flag_cols = [
    "psc_count",
    "has_corporate_psc",
    "has_foreign_psc",
    "has_sanctioned_psc",
    "recent_psc_change"
]

final_dataset[psc_flag_cols] = (
    final_dataset[psc_flag_cols]
    .fillna(0)      # companies with no PSCs
    .astype(int)    # convert back to integer
)

print(f"Join complete")
print(f"Final dataset rows: {len(final_dataset):,}")
print(f"Final dataset columns: {len(final_dataset.columns)}")

# ---------------------------------------------------------
# Save final Companies House dataset
# ---------------------------------------------------------

final_dataset.to_parquet(final_output_path, index=False)
# Ensure username is masked.
print(f"Final Companies House dataset saved to:\n  {mask_user(str(final_output_path))} on {date.today()}")
end = time.time()
print(f"Companies House data creation time: {end - start:.3f} seconds")

Loading processed FCDP and PSC feature tables...
FCDP rows: 5,677,276
PSC feature rows: 10,511,118
Join complete
Final dataset rows: 5,677,276
Final dataset columns: 13
Final Companies House dataset saved to:
  C:\Users\******\Documents\DS_Assignments\Companies_House\data_processed\company_psc_dataset.parquet on 2026-05-10
Companies House data creation time: 63.644 seconds


In [15]:
# Display the top 5 rows of the final dataset
final_dataset.head()

,company_number,company_category,company_age_when_acc_due,accounts_category,industry,country_of_origin,registered_country,psc_count,has_corporate_psc,has_foreign_psc,has_sanctioned_psc,recent_psc_change,overdue
0,1,Private Limited Company,9.204654,MICRO ENTITY,77299 - Renting and leasing of other personal ...,United Kingdom,ENGLAND,1,0,1,0,0,0
1,2,Community Interest Company,6.748802,TOTAL EXEMPTION FULL,62090 - Other information technology service a...,United Kingdom,ENGLAND,1,0,1,0,0,0
2,3,Private Limited Company,7.753593,UNAUDITED ABRIDGED,62012 - Business and domestic software develop...,United Kingdom,ENGLAND,3,0,1,0,0,0
3,4,Private Limited Company,8.774812,MICRO ENTITY,56102 - Unlicensed restaurants and cafes,United Kingdom,UNITED KINGDOM,2,0,1,0,0,1
4,5,Charitable Incorporated Organisation,NaN,NO ACCOUNTS FILED,None Supplied,United Kingdom,None,0,0,0,0,0,0
